<a href="https://colab.research.google.com/github/anaid3-hash/rnap-hack--equipo-7-RAKDBR-/blob/main/notebooks/entrenamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q transformers datasets accelerate scikit-learn pandas
!pip install -q transformers accelerate scikit-learn

In [11]:
from google.colab import files
uploaded = files.upload()

Saving train.tsv to train (2).tsv


In [12]:
import pandas as pd
import numpy as np
import torch

from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from datasets import Dataset

In [13]:
train_df = pd.read_csv("train.tsv", sep="\t")
test_df = pd.read_csv("test.tsv", sep="\t")
print(train_df.head())
print(train_df.columns)

   Unnamed: 0     id                                             prompt  \
0           0  12322                               you need to stop the   
1           1   1682  Article 11. On the basis of the information co...   
2           2  22592                           I have not been tweeting   
3           3  17390                                          NO-PROMPT   
4           4  30453                                          NO-PROMPT   

                                                text      label     model  \
0  you need to stop the engine and wait until it ...  generated         B   
1  The Commission shall publish the report; an in...  generated         C   
2  I have not been tweeting a lot lately, but I d...  generated         D   
3  I pass my exam and really thankgod for that bu...      human  NO-MODEL   
4  The template will have 3 parts: a mustache sha...      human  NO-MODEL   

   domain  
0  tweets  
1   legal  
2  tweets  
3  tweets  
4    wiki  
Index(['Unname

In [14]:
print("Columnas:")
print(train_df.columns.tolist())
print("\nEtiquetas:")
print(train_df["label"].value_counts())
print("\nValores únicos:")
print(train_df["label"].unique())


print("\nEjemplo de texto:")
print(train_df["text"].iloc[0])


Columnas:
['Unnamed: 0', 'id', 'prompt', 'text', 'label', 'model', 'domain']

Etiquetas:
label
human        17046
generated    16799
Name: count, dtype: int64

Valores únicos:
['generated' 'human']

Ejemplo de texto:
you need to stop the engine and wait until it stops. This is how I would do it: // Check if its safe


In [15]:
print("Dimensiones de entrenamiento:", train_df.shape)
print("Dimensiones de prueba:", test_df.shape)
print("\nDatos faltantes:")
print(train_df.isnull().sum())

Dimensiones de entrenamiento: (33845, 7)
Dimensiones de prueba: (21832, 7)

Datos faltantes:
Unnamed: 0    0
id            0
prompt        0
text          0
label         0
model         0
domain        0
dtype: int64


In [16]:
train_df = train_df[["text", "label"]].dropna()
test_df = test_df[["text", "label"]].dropna()

In [17]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL_NAME = "DeepESP/gpt2-spanish"
tokenizer_gpt = AutoTokenizer.from_pretrained(MODEL_NAME)
model_gpt = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_gpt.to(device)
model_gpt.eval()
print("GPT-2 cargado correctamente")
print("Dispositivo:", device)

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/840k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/499k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  261MB            

... (2 outputs left out)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

model.safetensors: downloading bytes:           |  0.00B            

[transformers] GPT2LMHeadModel LOAD REPORT from: DeepESP/gpt2-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 cargado correctamente
Dispositivo: cuda


In [18]:
import torch.nn.functional as F
import math
def calcular_perplejidad(texto):
   entradas = tokenizer_gpt(
       texto,
       return_tensors="pt",
       truncation=True,
       max_length=512
   )
   input_ids = entradas["input_ids"].to(device)
   attention_mask = entradas["attention_mask"].to(device)
   with torch.no_grad():
       outputs = model_gpt(input_ids)
       logits = outputs.logits[:, :-1, :]
       targets = input_ids[:, 1:]
       perdida = F.cross_entropy(
           logits.reshape(-1, logits.size(-1)),
           targets.reshape(-1),
           reduction="mean"
       )
       perplejidad = torch.exp(perdida).item()
   return perplejidad

In [19]:
texto = "Hola, buenos días. ¿Cómo puedo ayudarte?"
resultado = calcular_perplejidad(texto)
print("Perplejidad:", resultado)

Perplejidad: 14.75


In [20]:
textos = [
   "Hola, buenos días. ¿Cómo puedo ayudarte?",
   "Ayer fui al supermercado y olvidé comprar algunas cosas.",
   "El profe Jandro siempre anda a la moda.",
   "No manches, se me olvidó completamente hacer la tarea jajaja."
]
for texto in textos:
   p = calcular_perplejidad(texto)
   print(f"Texto: {texto}")
   print(f"Perplejidad: {p:.2f}\n")

Texto: Hola, buenos días. ¿Cómo puedo ayudarte?
Perplejidad: 14.75

Texto: Ayer fui al supermercado y olvidé comprar algunas cosas.
Perplejidad: 70.12

Texto: El profe Jandro siempre anda a la moda.
Perplejidad: 352.00

Texto: No manches, se me olvidó completamente hacer la tarea jajaja.
Perplejidad: 133.50



In [21]:
print("Etiquetas:")
print(train_df["label"].value_counts())
print("\nValores únicos:")
print(train_df["label"].unique())

Etiquetas:
label
human        17046
generated    16799
Name: count, dtype: int64

Valores únicos:
['generated' 'human']


In [22]:
from sklearn.model_selection import train_test_split
train_part, val_part = train_test_split(
   train_df,
   test_size=0.10,
   random_state=42,
   stratify=train_df["label"]
)
print("Entrenamiento:", len(train_part))
print("Validación:", len(val_part))
print("Prueba:", len(test_df))

Entrenamiento: 30460
Validación: 3385
Prueba: 21832


In [23]:

mapa = {
   "human": 0,
   "generated": 1
}
train_part = train_part.copy()
val_part = val_part.copy()
test_df = test_df.copy()
for df in [train_part, val_part, test_df]:
   df["label"] = df["label"].map(mapa)
print(train_part["label"].value_counts())
print(val_part["label"].value_counts())
print(test_df["label"].value_counts())

label
0    15341
1    15119
Name: count, dtype: int64
label
0    1705
1    1680
Name: count, dtype: int64
label
1    11190
0    10642
Name: count, dtype: int64


In [24]:
import time
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
tokenizer_gpt.pad_token = tokenizer_gpt.eos_token
tokenizer_gpt.padding_side = "right"
def calcular_perplejidades(textos, batch_size=16):
   resultados = []
   for inicio in tqdm(range(0, len(textos), batch_size)):
       lote = textos[inicio:inicio + batch_size]
       entradas = tokenizer_gpt(
           lote,
           return_tensors="pt",
           padding=True,
           truncation=True,
           max_length=256
       )
       input_ids = entradas["input_ids"].to(device)
       attention_mask = entradas["attention_mask"].to(device)
       with torch.no_grad():
           logits = model_gpt(input_ids).logits
           logits = logits[:, :-1, :]
           targets = input_ids[:, 1:]
           mask = attention_mask[:, 1:]
           perdidas = F.cross_entropy(
               logits.reshape(-1, logits.size(-1)),
               targets.reshape(-1),
               reduction="none"
           ).reshape(targets.shape)
           perdidas = perdidas * mask
           promedio = perdidas.sum(dim=1) / mask.sum(dim=1).clamp(min=1)
           ppl = torch.exp(promedio)
           resultados.extend(ppl.cpu().numpy())
   return np.array(resultados)

In [30]:
# Validación
inicio = time.time()
ppl_val = calcular_perplejidades(
    val_part["text"].tolist()
)
tiempo_val = time.time() - inicio  # <--- Correcto: tiempo final menos tiempo inicial

# Prueba
inicio = time.time()
ppl_test = calcular_perplejidades(
    test_df["text"].tolist()
)
tiempo_test = time.time() - inicio  # <--- Correcto: tiempo final menos tiempo inicial

print("Tiempo de validación:", tiempo_val)
print("Tiempo de prueba:", tiempo_test)

  0%|          | 0/212 [00:00<?, ?it/s]

  0%|          | 0/1365 [00:00<?, ?it/s]

Tiempo de validación: 17.72497844696045
Tiempo de prueba: 103.47286176681519


In [26]:
from sklearn.metrics import f1_score
y_val = val_part["label"].values
umbrales = np.quantile(
   ppl_val,
   np.linspace(0.01, 0.99, 200)
)
mejor_f1 = 0
mejor_umbral = None
for umbral in umbrales:
   pred = (ppl_val <= umbral).astype(int)
   f1 = f1_score(
       y_val,
       pred,
       average="macro"
   )
   if f1 > mejor_f1:
       mejor_f1 = f1
       mejor_umbral = umbral
print("Mejor umbral:", mejor_umbral)
print("F1 de validación:", mejor_f1)

Mejor umbral: 144.375
F1 de validación: 0.5755280224427676


In [27]:
from sklearn.metrics import (
   classification_report,
   confusion_matrix,
   f1_score,
   accuracy_score
)
y_test = test_df["label"].values
pred_test = (ppl_test <= mejor_umbral).astype(int)
print("F1 macro:", f1_score(
   y_test, pred_test, average="macro"
))
print("\nAccuracy:", accuracy_score(
   y_test, pred_test
))
print("\nReporte de clasificación:")
print(classification_report(
   y_test,
   pred_test,
   target_names=["Humano", "Máquina"]
))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred_test))

F1 macro: 0.6730347146130866

Accuracy: 0.6790491022352511

Reporte de clasificación:
              precision    recall  f1-score   support

      Humano       0.63      0.84      0.72     10642
     Máquina       0.77      0.53      0.63     11190

    accuracy                           0.68     21832
   macro avg       0.70      0.68      0.67     21832
weighted avg       0.70      0.68      0.67     21832


Matriz de confusión:
[[8893 1749]
 [5258 5932]]


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
import time

# 1. Medir tiempo y entrenar modelo base
inicio_base = time.time()

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(train_part["text"])
X_test_tfidf = vectorizer.transform(test_df["text"])

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_tfidf, train_part["label"])

tiempo_base = time.time() - inicio_base

# 2. Evaluar modelo base
pred_base = clf.predict(X_test_tfidf)
f1_base = f1_score(test_df["label"], pred_base, average="macro")
print(f"Modelo Base - F1 Macro: {f1_base:.4f} | Accuracy: {acc_base:.4f} | Tiempo: {tiempo_base:.2f}s")

print(f"Modelo Base - F1 Macro: {f1_base:.4f}, Tiempo: {tiempo_base:.2f}s")

Modelo Base - F1 Macro: 0.5215 | Accuracy: 0.5869 | Tiempo: 15.56s
Modelo Base - F1 Macro: 0.5215, Tiempo: 15.56s
